<div style="text-align: center;">

# Robotics and its Application (AI352) | Assignment 5

## Forward and Inverse Kinematics with Trajectory Generation

---

**Student Name:** *Naishadh Rana*

**Roll No:** U23CS014

---

</div>

## Part A: AI Integration

In [ ]:
import numpy as np
from math import sqrt, atan2, cos, sin, pi, degrees, radians
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from scipy.optimize import minimize
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import time

# Set link lengths
L1 = 3
L2 = 2
print("Libraries imported successfully!")

In [ ]:
def generate_dataset(n_samples=50000):
    X = []  # positions (x, y, z)
    Y = []  # joint angles (theta1, theta2, d)
    
    for _ in range(n_samples):
        theta1 = np.random.uniform(0, 2*np.pi)
        theta2 = np.random.uniform(-np.pi, np.pi)
        d = np.random.uniform(0.5, 2.0)  # prismatic extension
        
        # Forward kinematics
        x = L1 * np.cos(theta1) + L2 * np.cos(theta1 + theta2)
        y = L1 * np.sin(theta1) + L2 * np.sin(theta1 + theta2)
        z = d
        
        # Only keep points within workspace
        if np.sqrt(x**2 + y**2) <= (L1 + L2):
            X.append([x, y, z])
            Y.append([theta1, theta2, d])
    
    return np.array(X), np.array(Y)

X_data, Y_data = generate_dataset(50000)
print(f"Dataset size: {len(X_data)} samples")

In [ ]:
# Neural Network Class
class IKNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3, 64)  # Input: (x,y,z)
        self.fc2 = nn.Linear(64, 3)  # Output: (θ1, θ2, d)
    
    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

model = IKNet()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Convert to tensors
X_tensor = torch.FloatTensor(X_data)
Y_tensor = torch.FloatTensor(Y_data)
dataset = TensorDataset(X_tensor, Y_tensor)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

print("Model created")

In [ ]:
# Training loop
print("Training neural network...")
losses = []

for epoch in range(100):  # Using fewer epochs to be realistic
    epoch_loss = 0
    for batch_x, batch_y in dataloader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(dataloader)
    losses.append(avg_loss)
    
    if epoch % 10 == 0:
        print(f'Epoch {epoch}, Loss: {avg_loss:.4f}')

# Plot training progress
plt.figure(figsize=(8, 5))
plt.plot(losses)
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.show()

A simple feedforward network (3→64→3) was trained with MSE loss to learn inverse kinematics mapping (position → joint variables). Loss decreased steadily across epochs, indicating the model captured the nonlinear relationship.


In [ ]:
# Target position for comparison
target = np.array([1.2, 0.5, 0.7])
print(f"Target position: {target}")

print("\n3. Neural Network Method:")
model.eval()
with torch.no_grad():
    input_tensor = torch.FloatTensor(target.reshape(1, -1))
    start_time = time.time()
    prediction = model(input_tensor)
    nn_time = time.time() - start_time
    
    theta1_nn, theta2_nn, d_nn = prediction.numpy().flatten()
    
    # Check accuracy
    x_nn = L1 * np.cos(theta1_nn) + L2 * np.cos(theta1_nn + theta2_nn)
    y_nn = L1 * np.sin(theta1_nn) + L2 * np.sin(theta1_nn + theta2_nn)
    z_nn = d_nn
    predicted_pos = np.array([x_nn, y_nn, z_nn])
    nn_error = np.linalg.norm(predicted_pos - target)
    
    print(f"Joint angles: {np.degrees(theta1_nn):.2f}°, {np.degrees(theta2_nn):.2f}°, d={d_nn:.2f}")
    print(f"Error: {nn_error:.6f}")
    print(f"Time: {nn_time:.6f}s")

## Part D: Trajectory Generation for Smooth Motion

In [ ]:
## Task 1
# Robot constants
L1, L2 = 1.0, 1.0

def fk(theta1, theta2, L1=L1, L2=L2):
    return (L1*np.cos(theta1)+L2*np.cos(theta1+theta2),
            L1*np.sin(theta1)+L2*np.sin(theta1+theta2))

def ik_2r(x, y, L1=L1, L2=L2, elbow='up'):
    r2 = x*x + y*y
    r = np.sqrt(r2)
    if r > L1+L2 or r < abs(L1-L2):
        raise ValueError("Unreachable")
    c2 = (r2 - L1**2 - L2**2)/(2*L1*L2)
    c2 = np.clip(c2, -1, 1)
    theta2 = np.arccos(c2)
    if elbow == 'down':
        theta2 = -theta2
    alpha = np.arctan2(y, x)
    beta = np.arccos(np.clip((r2 + L1**2 - L2**2)/(2*L1*r), -1, 1))
    theta1 = alpha - beta if elbow == 'up' else alpha + beta
    return theta1, theta2

def generate_trajectory(q_start, q_end, n_steps=50, method='cubic', T=1.0):
    q_start = np.array(q_start); q_end = np.array(q_end)
    if method == 'linear':
        k = np.linspace(0,1,n_steps)[:,None]
        return q_start + k*(q_end - q_start)
    elif method == 'cubic':
        t = np.linspace(0, T, n_steps)
        a0 = q_start; a1 = 0
        a2 = 3*(q_end - q_start)/T**2
        a3 = -2*(q_end - q_start)/T**3
        return np.array([a0 + a1*ti + a2*ti**2 + a3*ti**3 for ti in t])
    else:
        raise ValueError("method must be 'linear' or 'cubic'")

In [ ]:
## Task 2: Trajectory Visualization

def make_pick_place_figure(traj, pick_pt, place_pt, home_angles=(0,0), interval=50):
    ee_path = np.array([fk(q[0], q[1]) for q in traj])

    fig = plt.figure(figsize=(14,4))
    ax_arm = fig.add_subplot(1,3,1)
    ax_arm.set_title("Arm Animation")
    ax_arm.set_xlim(-2.5,2.5); ax_arm.set_ylim(-2.5,2.5)
    ax_arm.set_aspect('equal'); ax_arm.grid()
    line, = ax_arm.plot([], [], 'o-', lw=2, label='Arm')
    # Markers
    home_xy = fk(home_angles[0], home_angles[1])
    ax_arm.scatter(*home_xy, c='gray', s=50, label='Home')
    ax_arm.scatter(*pick_pt, c='green', s=80, marker='*', label='Pick')
    ax_arm.scatter(*place_pt, c='red', s=80, marker='X', label='Place')
    ax_arm.legend(loc='upper left', fontsize=8)

    ax_angles = fig.add_subplot(1,3,2)
    ax_angles.set_title("Joint Angles")
    ax_angles.set_xlim(0, len(traj))
    ax_angles.set_ylim(-np.pi, np.pi)
    ax_angles.set_xlabel("Step"); ax_angles.set_ylabel("Angle (rad)")
    ax_angles.grid()
    th1_plot, = ax_angles.plot([], [], label='θ1')
    th2_plot, = ax_angles.plot([], [], label='θ2')
    ax_angles.legend()

    ax_path = fig.add_subplot(1,3,3)
    ax_path.set_title("End-Effector Path")
    ax_path.set_xlim(-2.5,2.5); ax_path.set_ylim(-2.5,2.5)
    ax_path.set_aspect('equal'); ax_path.grid()
    path_plot, = ax_path.plot([], [], 'r--')
    ax_path.scatter(pick_pt[0], pick_pt[1], c='green', marker='*')
    ax_path.scatter(place_pt[0], place_pt[1], c='red', marker='X')

    def update(i):
        q1, q2 = traj[i]
        x1 = L1*np.cos(q1); y1 = L1*np.sin(q1)
        x2, y2 = fk(q1, q2)
        line.set_data([0,x1,x2],[0,y1,y2])
        th1_plot.set_data(range(i+1), traj[:i+1,0])
        th2_plot.set_data(range(i+1), traj[:i+1,1])
        path_plot.set_data(ee_path[:i+1,0], ee_path[:i+1,1])
        return line, th1_plot, th2_plot, path_plot

    ani = animation.FuncAnimation(fig, update, frames=len(traj),
                                  interval=interval, blit=True, repeat=False)
    plt.tight_layout()
    return ani

In [ ]:
## Task 3: Pick and Place Operation Simulation
from IPython.display import HTML

home = (0.0, 0.0)
pick_pt = (1.5, 0.5)
place_pt = (-1.5, 0.5)

pick_angles = ik_2r(*pick_pt)         # (θ1, θ2)
place_angles = ik_2r(*place_pt)

traj1 = generate_trajectory(home, pick_angles, n_steps=50, method='linear', T=1.0)
traj2 = generate_trajectory(pick_angles, place_angles, n_steps=50, method='linear', T=1.0)
full_traj = np.vstack([traj1, traj2])

ani = make_pick_place_figure(full_traj, pick_pt, place_pt, home_angles=home, interval=50)
HTML(ani.to_jshtml())